# Exact seqFISH reanalysis used for final Fig. 2b

The analysis cells are copied from `251123-analysis_v2.ipynb`, which generated the current figure and Source Data. Only input and output paths are changed.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "analyses").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the Gao2026 code root")

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
GAO2026_DATA_ROOT = Path(
    os.environ.get("GAO2026_DATA_ROOT", REPO_ROOT / "data")
).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(
    os.environ.get("GAO2026_OUTPUT_ROOT", REPO_ROOT / "outputs")
).expanduser().resolve()


In [ ]:
# scripts/saESC_build_IF_allele_gene_table_ohishi.py
# -*- coding: utf-8 -*-
"""
Ohishi et al. (Sci Adv) データを Takei 解析と同様の形式に整形:
 - DNA/IF テーブル（アレル×geneID）から IF 強度を取得
 - RNA テーブルから (FOV,finalcellID,geneID) の ON セットと geneID→RNA_id を作る
 - geneID に RNA_id が付く行のみ解析対象にし、ON/OFF を付与
 - RNA_id × IF-marker ごとに gene-wise z を付与（ON/OFF 無視）
 - 併せて RNA_id ごとの ON 比率表を出力（遺伝子名は RNA_id を表示）
出力：Ohishi_et_al_seqFISH/results_saESC/ 配下
"""

from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

# ---------- 入出力 ----------
ROOT = GAO2026_DATA_ROOT / "04_seqFISH"
DNA_IF_CSV = ROOT / "Spot_coordinates_with_IF_intensity.csv"
RNA_ON_CSV = ROOT / "Transcriptio_state_summary.csv"

OUT_DIR = GAO2026_OUTPUT_ROOT / "04_seqFISH/results_saESC"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_OUT = OUT_DIR / "saESC_IF_allele_gene_RAW.csv.gz"
Z_OUT   = OUT_DIR / "saESC_IF_allele_gene_Z.csv.gz"
ONR_OUT = OUT_DIR / "saESC_gene_on_ratio.csv"

# ---------- 読み込み ----------
def read_csv_any(p: Path, usecols=None) -> pd.DataFrame:
    try:
        return pd.read_csv(p, usecols=usecols, engine="pyarrow")
    except Exception:
        return pd.read_csv(p, usecols=usecols)

# DNA/IF 側（IF 強度＋ geneID）
dna_cols_base = ["FOV", "finalcellID", "geneID"]
dna_df = read_csv_any(DNA_IF_CSV)
# 安全のため必要列だけ確実に残す
need = [c for c in dna_cols_base if c in dna_df.columns]
dna_df = dna_df[need + [c for c in dna_df.columns if c.endswith("_intensity")] +
                [c for c in ["x_int","y_int","z_int"] if c in dna_df.columns]].copy()

# IF マーカー列（*_intensity だが x_int,y_int,z_int は除外）
markers = [c for c in dna_df.columns
           if c.endswith("_intensity") and c not in {"x_int","y_int","z_int"}]
if not markers:
    raise RuntimeError("IF intensity 列（*_intensity）が見つかりません。")

# 型整理
for c in ["FOV","finalcellID"]:
    if c in dna_df.columns:
        dna_df[c] = pd.to_numeric(dna_df[c], errors="coerce").astype("Int64")
dna_df["geneID"] = dna_df["geneID"].astype(str)

# 便宜的なアレルキー（FOV と finalcellID の組合わせで一意）
dna_df["allele_key"] = "FOV" + dna_df["FOV"].astype(str) + "_finalcell" + dna_df["finalcellID"].astype(str)

# RNA 側（ON のみ列挙）
rna_cols = ["FOV", "finalcellID", "geneID", "RNA_id"]
rna_df = read_csv_any(RNA_ON_CSV, usecols=[c for c in rna_cols if c in pd.read_csv(RNA_ON_CSV, nrows=0).columns])
# 型
for c in ["FOV","finalcellID"]:
    if c in rna_df.columns:
        rna_df[c] = pd.to_numeric(rna_df[c], errors="coerce").astype("Int64")
rna_df["geneID"] = rna_df["geneID"].astype(str)
rna_df["RNA_id"] = rna_df["RNA_id"].astype("string").str.strip()

# ---------- geneID×RNA_id マッピング（1対多対応：DNA 側を複製） ----------
map_df = rna_df[["geneID", "RNA_id"]].copy()
map_df["RNA_id"] = map_df["RNA_id"].astype("string").str.strip()
map_df = map_df.dropna(subset=["RNA_id"]).drop_duplicates()
map_df = map_df[(map_df["RNA_id"] != "") & (map_df["RNA_id"].str.lower() != "nan")]

# geneID が複数の RNA_id に対応する場合、DNA 側の同一行が RNA_id の数だけ複製される
# （ユーザー要望：複数RNAに対応するデータは複数複製して扱う）
dna_df = dna_df.merge(map_df, on="geneID", how="inner")

# ---------- ON セット作成 (FOV, finalcellID, geneID) ----------
on_keys = set(zip(rna_df["FOV"].astype("Int64"),
                  rna_df["finalcellID"].astype("Int64"),
                  rna_df["geneID"].astype(str)))
dna_keys = list(zip(dna_df["FOV"].astype("Int64"),
                    dna_df["finalcellID"].astype("Int64"),
                    dna_df["geneID"].astype(str)))
dna_df["state"] = np.where([k in on_keys for k in dna_keys], "ON", "OFF")

# ---------- 要約：ON 比率（遺伝子ラベルは RNA_id を表示） ----------
base = dna_df[["RNA_id","allele_key","state"]].drop_duplicates()
total = base.groupby("RNA_id")["allele_key"].nunique().rename("n_alleles_dna")
oncnt = base[base["state"]=="ON"].groupby("RNA_id")["allele_key"].nunique().rename("n_alleles_on")
onrat = pd.concat([total, oncnt], axis=1).fillna(0.0).reset_index()
onrat["on_ratio"] = onrat["n_alleles_on"] / onrat["n_alleles_dna"].replace(0, np.nan)
onrat.rename(columns={"RNA_id":"RNA_id_label"}, inplace=True)
onrat.to_csv(ONR_OUT, index=False)

# ---------- gene-wise z（RNA_id×marker 毎に mean/std を計算し z 化；std=0 → 0） ----------
z_df = dna_df.copy()
for m in markers:
    stats = z_df.groupby("RNA_id")[m].agg(["mean","std"]).rename(columns={"mean":f"{m}__mean","std":f"{m}__std"})
    z_df = z_df.merge(stats, left_on="RNA_id", right_index=True, how="left")
    mu = z_df[f"{m}__mean"].astype(float)
    sd = z_df[f"{m}__std"].replace(0, np.nan).astype(float)
    z = (pd.to_numeric(z_df[m], errors="coerce") - mu) / sd
    z_df[f"{m}_z"] = z.fillna(0.0)
    z_df.drop(columns=[f"{m}__mean", f"{m}__std"], inplace=True)

# ---------- 保存 ----------
keep_cols_raw = ["FOV","finalcellID","allele_key","geneID","RNA_id","state"] + markers
keep_cols_raw += [c for c in ["x_int","y_int","z_int"] if c in z_df.columns]  # あれば

z_cols = ["FOV","finalcellID","allele_key","geneID","RNA_id","state"] + [f"{m}_z" for m in markers]

dna_df[keep_cols_raw].to_csv(RAW_OUT, index=False, compression="gzip")
z_df[z_cols].to_csv(Z_OUT, index=False, compression="gzip")

print(f"[done] RAW: {RAW_OUT}")
print(f"[done] Z  : {Z_OUT}")
n_genes = z_df["RNA_id"].nunique()
n_rows  = z_df.shape[0]
n_on    = int((z_df["state"]=="ON").sum())
n_off   = int((z_df["state"]=="OFF").sum())
print(f"[info] genes(RNA_id)={n_genes}  rows(allele×gene)={n_rows:,}  ON={n_on:,}  OFF={n_off:,}")
print(f"[info] markers({len(markers)}): {markers[:8]} ...")
print(f"[info] on-ratio table: {ONR_OUT}")

In [ ]:
# saESC（Ohishi et al.）: RNA_id が付く領域のみで Δ を作図
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# ===== パス =====
BASE = GAO2026_DATA_ROOT / "04_seqFISH"
DNA_IF_CSV = BASE / "Spot_coordinates_with_IF_intensity.csv"
RNA_ON_CSV = BASE / "Transcriptio_state_summary.csv"
OUT_DIR = GAO2026_OUTPUT_ROOT / "04_seqFISH/results_saESC_delta"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 解析パラメータ
ON_RATIO_MIN = 0.00001    # 0.001%
DPI = 220
H_PER = 0.22           # 1 マーカーあたりの縦幅

def p_to_stars(p: float) -> str:
    if not np.isfinite(p): return "ns"
    if p <= 1e-4: return "****"
    if p <= 1e-3: return "***"
    if p <= 1e-2: return "**"
    if p <= 0.05: return "*"
    return "ns"

# ========== 1) 入力読み込み & 前処理 ==========
# DNA/IF（全アレル×領域; intensity 列を含む）
dna = pd.read_csv(DNA_IF_CSV)
# RNA（ON のみが列挙）
rna = pd.read_csv(RNA_ON_CSV)

# 型・列名の整え
for c in ["FOV","cell","finalcellID"]:
    if c in dna.columns: dna[c] = pd.to_numeric(dna[c], errors="coerce").astype("Int64")
    if c in rna.columns: rna[c] = pd.to_numeric(rna[c], errors="coerce").astype("Int64")

dna["geneID"] = dna["geneID"].astype(str)
rna["geneID"] = rna["geneID"].astype(str)
rna["RNA_id"] = rna["RNA_id"].astype("string").str.strip()

# 解析に使う一意アレルキー（finalcellID はグローバル一意ではないので、FOV+cell と組にします）
def make_allele_key(df):
    return ("FOV"+df["FOV"].astype(str) + "_cell"+df["cell"].astype(str) + "_final"+df["finalcellID"].astype(str))

dna["allele_key"] = make_allele_key(dna)
rna["allele_key"] = make_allele_key(rna)

# intensity 列を抽出
int_cols = [c for c in dna.columns if c.endswith("_intensity")]
if not int_cols:
    raise RuntimeError("IF intensity 列（*_intensity）が見つかりません。")

# -------- RNA_id の付く geneID だけに制限（1対多対応：DNA 側を複製） --------
map_df = rna[["geneID", "RNA_id"]].copy()
map_df["RNA_id"] = map_df["RNA_id"].astype("string").str.strip()
map_df = map_df.dropna(subset=["RNA_id"]).drop_duplicates()
map_df = map_df[(map_df["RNA_id"] != "") & (map_df["RNA_id"].str.lower() != "nan")]

# geneID が複数の RNA_id に対応する場合、DNA 側の同一行が RNA_id の数だけ複製される
# （ユーザー要望：複数RNAに対応するデータは複数複製して扱う）
dna = dna.merge(map_df, on="geneID", how="inner")

# -------- ON/OFF 付与（RNA: ON のみ列挙 → DNA 側に合流）--------
on_keys = set((rna["allele_key"].astype(str) + "|" + rna["geneID"]).tolist())
dna["state"] = np.where((dna["allele_key"].astype(str) + "|" + dna["geneID"]).isin(on_keys), "ON", "OFF")

# 解析の母集団は「DNA で観測された allele_key×geneID」（= dna の行）
# RNA_id 名で集計するためキーを RNA_id に統一
dna["GENE"] = dna["RNA_id"].astype(str)

# ========== 2) 遺伝子ごとに gene-wise z を計算 ==========
# 遺伝子×マーカーごとの mean/std で正規化（state は無視）
stats = (dna.groupby("GENE")[int_cols]
           .agg(["mean","std"]))
# 列を（marker）→ ("marker","mean"),("marker","std") → dict で引けるよう整形
means = stats.xs("mean", level=1, axis=1)
stds  = stats.xs("std",  level=1, axis=1)
stds  = stds.replace(0, np.nan)

# 値を引き当てて z に変換
def z_norm_block(df_block: pd.DataFrame) -> pd.DataFrame:
    g = df_block["GENE"].iloc[0]
    out = df_block[int_cols].copy()
    mu = means.loc[g, int_cols] if g in means.index else np.nan
    sd = stds.loc[g, int_cols]  if g in stds.index  else np.nan
    out = (out - mu) / sd
    return out.fillna(0.0)

z_values = (dna.groupby("GENE", group_keys=False)
              .apply(z_norm_block))
z_values.columns = [c.replace("_intensity", "_z") for c in z_values.columns]
dna_z = pd.concat([dna[["GENE","RNA_id","geneID","allele_key","state"]], z_values], axis=1)

# 保存（再利用用）
raw_out = OUT_DIR / "saESC_IF_allele_gene_RAW.csv.gz"
z_out   = OUT_DIR / "saESC_IF_allele_gene_Z.csv.gz"
dna.to_csv(raw_out, index=False, compression="gzip")
dna_z.to_csv(z_out,   index=False, compression="gzip")

# ========== 3) ON 割合（母集団＝DNA 観測アレル）で 10% 以上の遺伝子だけ採用 ==========
base = dna_z[["GENE","allele_key","state"]].drop_duplicates()
tot = base.groupby("GENE")["allele_key"].nunique().rename("n_alleles_dna")
onc = base[base["state"]=="ON"].groupby("GENE")["allele_key"].nunique().rename("n_alleles_on")
onratio = (pd.concat([tot, onc], axis=1)
             .fillna(0.0)
             .assign(on_ratio=lambda d: d["n_alleles_on"]/d["n_alleles_dna"])
             .reset_index()
             .sort_values("on_ratio", ascending=False))
onratio_out = OUT_DIR / "saESC_gene_on_ratio_by_RNAid.csv"
onratio.to_csv(onratio_out, index=False)

keep_genes = onratio.loc[onratio["on_ratio"] >= ON_RATIO_MIN, "GENE"].astype(str).tolist()
print(f"[info] genes kept by ON≥10%: {len(keep_genes)}  (of {onratio.shape[0]})")

dna_z_sel = dna_z[dna_z["GENE"].isin(keep_genes)].copy()

# ========== 4) Δ = median(ON) − median(OFF) を遺伝子ごとに算出 ==========
# long 化
z_cols = [c for c in dna_z_sel.columns if c.endswith("_z")]
long = dna_z_sel.melt(id_vars=["GENE","state"], value_vars=z_cols,
                      var_name="marker", value_name="z")

# 遺伝子×状態×マーカーの中央値
med = (long.groupby(["GENE","state","marker"], as_index=False)["z"]
            .median().rename(columns={"z":"z_median"}))

# Δ（ON−OFF）を遺伝子×マーカーで
med_on  = med[med["state"]=="ON"][["GENE","marker","z_median"]].rename(columns={"z_median":"z_on"})
med_off = med[med["state"]=="OFF"][["GENE","marker","z_median"]].rename(columns={"z_median":"z_off"})
delta = (med_on.merge(med_off, on=["GENE","marker"], how="inner")
               .assign(delta=lambda d: d["z_on"] - d["z_off"]))

delta.to_csv(OUT_DIR / "saESC_delta_per_gene_marker.csv.gz", index=False, compression="gzip")

# ========== 5) マーカーごとに Δ のボックスプロット ==========
# マーカー並びは pooled Δ 中央値（大→小）
order = (delta.groupby("marker")["delta"].median()
               .sort_values(ascending=False).index.tolist())

vals = [pd.to_numeric(delta.loc[delta["marker"]==m, "delta"], errors="coerce").dropna().values
        for m in order]

# x 軸スケール：|Δ| の 99% パーセンタイルから決定
flat = pd.to_numeric(delta["delta"], errors="coerce").dropna().values
xmax = 0.25
if flat.size:
    xmax = float(np.nanpercentile(np.abs(flat), 99))
    xmax = max(0.1, np.ceil((xmax + 0.02)*10)/10)
xlim = (-xmax*1.2, xmax*1.3)

fig_h = max(6.0, H_PER * len(order))
fig, ax = plt.subplots(figsize=(9.5, fig_h))
y = np.arange(len(order))

ax.set_facecolor("white")
ax.grid(axis="x", color="#DDDDDD", lw=0.8)
ax.grid(axis="y", color="#EEEEEE", lw=0.6)
ax.axvline(0.0, color="#888888", lw=1.0, ls="--", zorder=0)

ax.boxplot(vals, vert=False, positions=y, widths=0.55,
           showcaps=True, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor="#6AA5FF", alpha=0.35, edgecolor="#2B5CB8"),
           medianprops=dict(color="black"),
           whiskerprops=dict(color="#2B5CB8"),
           capprops=dict(color="#2B5CB8"))

# 1 標本 Wilcoxon（Δ vs 0）→ 星印（行右端）
x_star = xlim[1] - 0.03*(xlim[1]-xlim[0])
stars = []
for i, m in enumerate(order):
    v = pd.to_numeric(delta.loc[delta["marker"]==m, "delta"], errors="coerce").dropna().values
    if v.size >= 1 and np.any(v!=0):
        try:
            _, p = wilcoxon(v, alternative="two-sided", zero_method="wilcox")
        except Exception:
            p = np.nan
    else:
        p = np.nan
    ax.text(x_star, i, p_to_stars(p), ha="right", va="center", fontsize=9)
    stars.append((m, p, float(np.median(v)) if v.size else np.nan))

ax.set_xlim(*xlim)
ax.set_yticks(y)
# マーカー名は *_z を外してすっきり表示
ax.set_yticklabels([m.replace("_z", "") for m in order], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Δ = median(ON) − median(OFF)  (gene-wise z)", fontsize=11)
ax.set_title("saESC — Δ boxplots per IF marker (genes with ON≥10%)", fontsize=12)

fig.tight_layout()
png_out = OUT_DIR / "saESC_delta_boxplots_ONge10.png"
fig.savefig(png_out, dpi=DPI, bbox_inches="tight")
plt.close(fig)

# 検定表も保存
pd.DataFrame({"marker":[m for m,_,_ in stars],
              "wilcoxon_p":[p for _,p,_ in stars],
              "median_delta":[med for _,_,med in stars]}
            ).to_csv(OUT_DIR / "saESC_delta_stats.csv", index=False)

print(f"[done] RAW : {raw_out}")
print(f"[done] Z   : {z_out}")
print(f"[done] ON ratio table : {onratio_out}")
print(f"[done] Δ per gene/marker : {OUT_DIR/'saESC_delta_per_gene_marker.csv.gz'}")
print(f"[done] Figure : {png_out}")

In [ ]:
# Δz by on_ratio quartiles: boxplot + all points + BH(FDR) stars
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# ----------------------------
# Inputs (prefer existing notebook variables; fall back to file paths)
# ----------------------------
BASE_ = globals().get("BASE", None)
if BASE_ is None:
    BASE_ = Path("../Ohishi_et_al_seqFISH")

IN_ONR_ = globals().get("IN_ONR", None)
if IN_ONR_ is None:
    IN_ONR_ = GAO2026_OUTPUT_ROOT / "04_seqFISH/results_saESC_delta/saESC_gene_on_ratio_by_RNAid.csv"

DELTA_CSV_ = globals().get("DELTA_CSV", None)
if DELTA_CSV_ is None:
    DELTA_CSV_ = GAO2026_OUTPUT_ROOT / "04_seqFISH/results_saESC_delta/saESC_delta_per_gene_marker.csv.gz"

OUT_ = GAO2026_OUTPUT_ROOT / "04_seqFISH/outputs"
OUT_.mkdir(parents=True, exist_ok=True)

DPI_ = int(globals().get("DPI", 220))

# ----------------------------
# Helpers
# ----------------------------
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    msk = np.isfinite(pvals)
    if not np.any(msk):
        return qvals
    p = pvals[msk]
    m = p.size
    order = np.argsort(p)
    ranked = p[order]
    ranks = np.arange(1, m + 1, dtype=float)
    q = ranked * m / ranks
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0.0, 1.0)
    q_back = np.empty_like(q)
    q_back[order] = q
    qvals[msk] = q_back
    return qvals

def stars_from_q(q: float) -> str:
    if not np.isfinite(q):
        return "NA"
    if q <= 1e-4:
        return "****"
    if q <= 1e-3:
        return "***"
    if q <= 1e-2:
        return "**"
    if q <= 5e-2:
        return "*"
    return "ns"

def safe_wilcoxon_vs0(values: np.ndarray) -> float:
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return np.nan
    if not np.any(v != 0):
        return np.nan
    try:
        _, p = wilcoxon(v, alternative="two-sided", zero_method="wilcox")
        return float(p)
    except Exception:
        return np.nan

# ----------------------------
# Load tables
# ----------------------------
onr2 = pd.read_csv(IN_ONR_)
if "RNA_id" in onr2.columns and "GENE" not in onr2.columns:
    onr2 = onr2.rename(columns={"RNA_id": "GENE"})
if "GENE" not in onr2.columns:
    raise RuntimeError(f"IN_ONR is missing required column 'GENE': {IN_ONR_}")
if "on_ratio" not in onr2.columns:
    raise RuntimeError(f"IN_ONR is missing required column 'on_ratio': {IN_ONR_}")
onr2 = onr2[["GENE", "on_ratio"]].copy()
onr2["GENE"] = onr2["GENE"].astype(str)
onr2["on_ratio"] = pd.to_numeric(onr2["on_ratio"], errors="coerce")
onr2 = onr2.dropna(subset=["GENE", "on_ratio"]).copy()

dlt = pd.read_csv(DELTA_CSV_)
need_cols = {"GENE", "marker", "delta"}
if not need_cols.issubset(dlt.columns):
    raise RuntimeError(f"DELTA_CSV is missing required columns {sorted(need_cols)}: columns={list(dlt.columns)}")
dlt = dlt[["GENE", "marker", "delta"]].copy()
dlt["GENE"] = dlt["GENE"].astype(str)
dlt["marker"] = dlt["marker"].astype(str)
dlt["delta"] = pd.to_numeric(dlt["delta"], errors="coerce")
dlt = dlt.dropna(subset=["GENE", "marker", "delta"]).copy()

# ----------------------------
# Quartile assignment (based on ALL genes in onr)
# ----------------------------
rank_tbl = (onr2.sort_values(["on_ratio", "GENE"], ascending=[False, True])
              .reset_index(drop=True))
G = int(rank_tbl.shape[0])
if G < 4:
    raise RuntimeError(f"Too few genes (G={G}).")
rank_tbl["rank"] = np.arange(1, G + 1, dtype=int)
q_idx = np.floor((rank_tbl["rank"].to_numpy() - 1) / G * 4).astype(int)
q_idx = np.clip(q_idx, 0, 3)
q_labels = np.array(["0-25", "25-50", "50-75", "75-100"], dtype=object)
rank_tbl["quartile"] = q_labels[q_idx]
gene2q = rank_tbl.set_index("GENE")["quartile"].to_dict()

plot_df = dlt.copy()
plot_df["quartile"] = plot_df["GENE"].map(gene2q)
plot_df = plot_df.dropna(subset=["quartile"]).copy()
plot_df["quartile"] = pd.Categorical(plot_df["quartile"], categories=["0-25", "25-50", "50-75", "75-100"], ordered=True)

# Marker order: pooled median Δ (desc), same as existing notebook
marker_order = (plot_df.groupby("marker")["delta"].median()
                    .sort_values(ascending=False).index.tolist())
plot_df["marker"] = pd.Categorical(plot_df["marker"], categories=marker_order, ordered=True)

# ----------------------------
# Stats: Wilcoxon vs 0 per (marker × quartile) + BH-FDR
# ----------------------------
stats_rows = []
for m in marker_order:
    for q in ["0-25", "25-50", "50-75", "75-100"]:
        vals = plot_df[(plot_df["marker"] == m) & (plot_df["quartile"] == q)]["delta"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        p = safe_wilcoxon_vs0(vals)
        stats_rows.append({"marker": m, "quartile": q, "n": int(vals.size), "pvalue": p})
stats_df = pd.DataFrame(stats_rows)
stats_df["qvalue_bh"] = bh_fdr(stats_df["pvalue"].to_numpy(dtype=float))
stats_df["stars"] = [stars_from_q(q) for q in stats_df["qvalue_bh"].to_numpy(dtype=float)]

# ----------------------------
# Plot: boxplot + all points (same color)
# ----------------------------
plt.rcParams.update({"font.family": "Arial", "pdf.fonttype": 42, "ps.fonttype": 42})

M = len(marker_order)
quartiles = ["0-25", "25-50", "50-75", "75-100"]

# N = number of genes per quartile (count only genes present in both dlt and onr)
q_counts = (
    plot_df[["GENE", "quartile"]]
    .drop_duplicates()
    .groupby("quartile")
    .size()
    .reindex(quartiles, fill_value=0)
)

q_label_map = {q: f"{q} (N = {int(q_counts[q])})" for q in quartiles}
quartile_labels = [q_label_map[q] for q in quartiles]

offsets = np.linspace(-0.30, 0.30, len(quartiles))
box_w = 0.12
pt_color = "#333333"
pt_alpha = 0.55
pt_size = 9
rng = np.random.default_rng(0)

all_vals = plot_df["delta"].to_numpy(dtype=float)
all_vals = all_vals[np.isfinite(all_vals)]
if all_vals.size == 0:
    raise RuntimeError("No valid Δz values (all NaN).")
ymin = float(np.nanpercentile(all_vals, 1))
ymax = float(np.nanpercentile(all_vals, 99))
yr = ymax - ymin
if not np.isfinite(yr) or yr <= 0:
    yr = float(np.nanmax(all_vals) - np.nanmin(all_vals))
    if not np.isfinite(yr) or yr <= 0:
        yr = 1.0

#fig_w = max(10.0, 0.33 * M + 6.0)
fig_w = 7.5
fig_h = 4
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_facecolor("white")
ax.grid(axis="y", color="#DDDDDD", lw=0.8)
ax.axhline(0.0, color="#888888", lw=1.0, ls="--", zorder=0)

base_x = np.arange(M, dtype=float)
positions = []
data_for_boxes = []
meta = []  # (marker, quartile, pos)
for i, m in enumerate(marker_order):
    for j, q in enumerate(quartiles):
        pos = base_x[i] + offsets[j]
        vals = plot_df[(plot_df["marker"] == m) & (plot_df["quartile"] == q)]["delta"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue
        positions.append(pos)
        data_for_boxes.append(vals)
        meta.append((m, q, pos, vals))

# Added: quartile colors (take 4 colors from a matplotlib colormap)
# cmap_q = plt.cm.get_cmap("Set2", len(quartiles))
# q2color = {q: cmap_q(i) for i, q in enumerate(quartiles)}

q2color = {
    "0-25":   "#0072B2",  # blue
    "25-50":  "#E69F00",  # orange
    "50-75":  "#009E73",  # green
    "75-100": "#CC79A7",  # magenta
}

bp = ax.boxplot(
    data_for_boxes,
    positions=positions,
    widths=box_w,
    showfliers=True,  # <- set to True
    patch_artist=True,
    manage_ticks=False,
    boxprops=dict(facecolor="white", edgecolor="black", linewidth=0.8),
    medianprops=dict(color="black", linewidth=1.0),
    whiskerprops=dict(color="black", linewidth=0.8),
    capprops=dict(color="black", linewidth=0.8),
    flierprops=dict(  # <- outlier marker style
        marker="o",
        markerfacecolor="#333333",   # marker fill
        markeredgecolor="#333333",
        markersize=3,
        linestyle="none",
        alpha=0.8,
        markeredgewidth=0.7,
    ),
    zorder=2,)

# Added: color boxes by quartile (meta order == boxes order)
for k, (_, q, _, _) in enumerate(meta):
    b = bp["boxes"][k]
    b.set_facecolor(q2color[q])
    b.set_alpha(0.55)
    b.set_edgecolor("black")

# overlay all points (same color) + stars
for (m, q, pos, vals) in meta:
    xj = rng.normal(loc=pos, scale=0.02, size=vals.size)
    #ax.scatter(xj, vals, s=pt_size, c=pt_color, alpha=pt_alpha, linewidths=0, zorder=3)
    # ax.scatter(
    #     xj, vals,
    #     s=pt_size,
    #     facecolors="none",          # no fill
    #     edgecolors=pt_color,        # edge color
    #     linewidths=0.3,             # edge linewidth
    #     alpha=0.8,                  # edge alpha (as needed)
    #     marker="o",
    #     zorder=3,
    # )
    row = stats_df[(stats_df["marker"] == m) & (stats_df["quartile"] == q)]
    if not row.empty:
        star = str(row["stars"].iloc[0])
        if star not in ("ns", "NA"):
            y_star = float(np.nanmax(vals)) + 0.06 * yr
            y_star += (quartiles.index(q)) * 0.015 * yr
            ax.text(pos, y_star, star, ha="center", va="bottom", fontsize=11, color="black", zorder=4)

# x ticks: IF target names (no rotation, horizontal)
ax.set_xticks(base_x)
ax.set_xticklabels(
    [str(m).replace("_z", "").replace("_intensity", "") for m in marker_order],
    rotation=0,
    ha="center",
)
ax.tick_params(axis="x", labelsize=8)

ax.set_xlabel("IF targets")
ax.set_ylabel("Δz = median(z|Active) − median(z|Inactive)")

# legend (top, colored by quartile)
from matplotlib.lines import Line2D


handles = [
    Line2D(
        [0],[0],
        color="black", lw=0,
        marker="s", markersize=8,
        markerfacecolor=q2color[q],
        markeredgecolor="black",
        label=q_label_map[q],   # <- label includes N per quartile
    )
    for q in quartiles
]

ax.legend(
    handles=handles,
    title="Active ratio rank quartile (%)",
    frameon=False,
    ncol=4,
    loc="lower center",          # anchor legend box by its lower center
    bbox_to_anchor=(0.5, 1.02),  # slightly above the axes (axes coordinates)
    borderaxespad=0.0,
)

ax.set_title("")
fig.tight_layout()

out_png = OUT_ / "saESC_delta_boxplot_onratio_quartiles_points.png"
out_pdf = OUT_ / "saESC_delta_boxplot_onratio_quartiles_points.pdf"
#fig.savefig(out_png, dpi=DPI_, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
plt.close(fig)

# Save stats table for reference
stats_out = OUT_ / "saESC_delta_wilcoxon_bh_onratio_quartiles.csv"
stats_df.to_csv(stats_out, index=False)

print("[done] plot:", out_png)
print("[done] plot:", out_pdf)
print("[done] stats:", stats_out)
display(stats_df.sort_values(["marker", "quartile"]))

In [ ]:
# Active (ON) allele count per gene by on_ratio quartiles: boxplot + outliers, save PDF
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------
# Inputs (reuse existing variables when present)
# ----------------------------
BASE__ = globals().get("BASE", None)
if BASE__ is None:
    BASE__ = Path("../Ohishi_et_al_seqFISH")

IN_ONR__ = globals().get("IN_ONR", None)
if IN_ONR__ is None:
    IN_ONR__ = GAO2026_OUTPUT_ROOT / "04_seqFISH/results_saESC_delta/saESC_gene_on_ratio_by_RNAid.csv"

OUT__ = GAO2026_OUTPUT_ROOT / "04_seqFISH/outputs"
OUT__.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Load on-ratio table (must contain n_alleles_on)
# ----------------------------
onr_full = pd.read_csv(IN_ONR__)
if "RNA_id" in onr_full.columns and "GENE" not in onr_full.columns:
    onr_full = onr_full.rename(columns={"RNA_id": "GENE"})
need_cols = {"GENE", "on_ratio", "n_alleles_on"}
if not need_cols.issubset(onr_full.columns):
    raise RuntimeError(f"ON ratio table needs columns {sorted(need_cols)} but got {list(onr_full.columns)}: {IN_ONR__}")

onr_full = onr_full[["GENE", "on_ratio", "n_alleles_on"]].copy()
onr_full["GENE"] = onr_full["GENE"].astype(str)
onr_full["on_ratio"] = pd.to_numeric(onr_full["on_ratio"], errors="coerce")
onr_full["n_alleles_on"] = pd.to_numeric(onr_full["n_alleles_on"], errors="coerce")
onr_full = onr_full.dropna(subset=["GENE", "on_ratio", "n_alleles_on"]).copy()

# ----------------------------
# Quartile assignment by on_ratio rank (population = all genes in onr table)
# ----------------------------
rank_tbl2 = (onr_full.sort_values(["on_ratio", "GENE"], ascending=[False, True])
               .reset_index(drop=True))
G2 = int(rank_tbl2.shape[0])
if G2 < 4:
    raise RuntimeError(f"Too few genes in on-ratio table (G={G2}).")
rank_tbl2["rank"] = np.arange(1, G2 + 1, dtype=int)
q_idx2 = np.floor((rank_tbl2["rank"].to_numpy() - 1) / G2 * 4).astype(int)
q_idx2 = np.clip(q_idx2, 0, 3)
quartiles = ["0-25", "25-50", "50-75", "75-100"]
q_labels2 = np.array(quartiles, dtype=object)
rank_tbl2["quartile"] = q_labels2[q_idx2]
gene2q2 = rank_tbl2.set_index("GENE")["quartile"].to_dict()

dfq = onr_full.copy()
dfq["quartile"] = dfq["GENE"].map(gene2q2)
dfq = dfq.dropna(subset=["quartile"]).copy()
dfq["quartile"] = pd.Categorical(dfq["quartile"], categories=quartiles, ordered=True)

# N (genes) per quartile for labeling
q_counts2 = (dfq[["GENE", "quartile"]]
               .drop_duplicates()
               .groupby("quartile")
               .size()
               .reindex(quartiles, fill_value=0))
quartile_labels2 = [f"{q} \n(N = {int(q_counts2[q])})" for q in quartiles]

# Data for boxplot (n_alleles_on per gene)
box_data = []
for q in quartiles:
    v = dfq.loc[dfq["quartile"] == q, "n_alleles_on"].to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    box_data.append(v)

# ----------------------------
# Plot (match the style of the last cell)
# ----------------------------
plt.rcParams.update({"font.family": "Arial", "pdf.fonttype": 42, "ps.fonttype": 42})
fig, ax = plt.subplots(figsize=(3.5, 3.5*1.5))
ax.set_facecolor("white")
ax.grid(axis="y", color="#DDDDDD", lw=0.8)
ax.set_axisbelow(True)

# cmap_q2 = plt.cm.get_cmap("Set2", len(quartiles))
# q2color2 = {q: cmap_q2(i) for i, q in enumerate(quartiles)}

q2color2 = {
    "0-25":   "#0072B2",  # blue
    "25-50":  "#E69F00",  # orange
    "50-75":  "#009E73",  # green
    "75-100": "#CC79A7",  # magenta
}

bp2 = ax.boxplot(
    box_data,
    positions=np.arange(len(quartiles), dtype=float),
    widths=0.55,
    showfliers=True,
    patch_artist=True,
    boxprops=dict(facecolor="white", edgecolor="black", linewidth=0.8),
    medianprops=dict(color="black", linewidth=1.0),
    whiskerprops=dict(color="black", linewidth=0.8),
    capprops=dict(color="black", linewidth=0.8),
    flierprops=dict(
        marker="o",
        markerfacecolor="#333333",
        markeredgecolor="#333333",
        markersize=3,
        linestyle="none",
        alpha=0.8,
        markeredgewidth=0.7,
    ),
    zorder=2,
 )

# Color boxes by quartile
for i, q in enumerate(quartiles):
    bp2["boxes"][i].set_facecolor(q2color2[q])
    bp2["boxes"][i].set_alpha(0.55)
    bp2["boxes"][i].set_edgecolor("black")

# X axis labels
ax.set_xticks(np.arange(len(quartiles), dtype=float))
ax.set_xticklabels(quartile_labels2, rotation=0, ha="center")
ax.tick_params(axis="x", labelsize=8)

ax.set_xlabel("Active ratio rank quartile (%)")
ax.set_ylabel("Active allele count per gene")
ax.set_title("")

# Legend (top, 4 columns)
from matplotlib.lines import Line2D
handles2 = [
    Line2D([0], [0], lw=0, marker="s", markersize=8,
           markerfacecolor=q2color2[q], markeredgecolor="black",
           label=quartile_labels2[i])
    for i, q in enumerate(quartiles)
 ]
# ax.legend(
#     handles=handles2,
#     title="on_ratio rank quartile (%)",
#     frameon=False,
#     ncol=4,
#     loc="lower center",
#     bbox_to_anchor=(0.5, 1.02),
#     borderaxespad=0.0,
#  )

fig.tight_layout(rect=[0, 0, 1, 0.92])

out_pdf2 = OUT__ / "saESC_active_allelecount_by_onratio_quartiles_boxplot.pdf"
fig.savefig(out_pdf2, bbox_inches="tight")
plt.close(fig)

print("[done] plot:", out_pdf2)